<a href="https://colab.research.google.com/github/fatmasenguler/Spanning-Tree_Thermostatics_of_Allostery/blob/main/4_participation_ratio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 8.5 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 6GOD.pdb to 6GOD.pdb
Saving 6GOF.pdb to 6GOF.pdb


In [ ]:
"""
4_participation_ratio.py
========================
Computes the Participation Ratio (PR) for five KRAS allosteric channels
in wild-type (6GOD) and G12D mutant (6GOF) ensembles.

PR = 1 / sum_i(p_i^2)

where p_i is the normalized Burton-Pemantle residue probability.

Dependencies: numpy (standalone, Colab-compatible)

Usage:
    python 4_participation_ratio.py
    # Place 6GOD.pdb and 6GOF.pdb in working directory

Author: Fatma Ciftci & Burak Erman
"""

import numpy as np
from collections import defaultdict
import os

CUTOFF = 7.8
KT = 1.0
MAX_LEN = 9

CHANNELS = [
    (6, 11,   "Ch1: P-loop (6-11)"),
    (55, 60,  "Ch2: pre-Switch II (55-60)"),
    (110, 117, "Ch3: GBS (110-117)"),
    (141, 146, "Ch4: SAK motif (141-146)"),
    (19, 142,  "Ch5: Inter-lobe linker (19-142)"),
]


def parse_pdb_ca(filename):
    coords = {}
    with open(filename) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                atom_name = line[12:16].strip()
                if atom_name == "CA":
                    resid = int(line[22:26].strip())
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    if resid not in coords:
                        coords[resid] = np.array([x, y, z])
    return coords


def build_contact_graph(coords, cutoff=7.8, kT=1.0):
    residues = sorted(coords.keys())
    adj = defaultdict(dict)
    for i, ri in enumerate(residues):
        for j in range(i + 1, len(residues)):
            rj = residues[j]
            d = np.linalg.norm(coords[ri] - coords[rj])
            if d <= cutoff:
                w = np.exp(-d / kT)
                adj[ri][rj] = w
                adj[rj][ri] = w
    nodes = sorted(set(adj.keys()))
    return adj, nodes


def build_laplacian(adj, nodes):
    n = len(nodes)
    node_idx = {r: i for i, r in enumerate(nodes)}
    L = np.zeros((n, n))
    for ri in nodes:
        for rj, w in adj[ri].items():
            if rj in node_idx:
                i, j = node_idx[ri], node_idx[rj]
                L[i, j] -= w
                L[i, i] += w
    Lplus = np.linalg.pinv(L)
    return L, Lplus, node_idx


def enumerate_paths(adj, source, target, max_len):
    if source not in adj or target not in adj:
        return []
    all_paths = []
    stack = [(source, [source])]
    while stack:
        node, path = stack.pop()
        if len(path) > max_len:
            continue
        if node == target and len(path) > 1:
            all_paths.append(path[:])
            continue
        for neighbor in adj[node]:
            if neighbor not in path and len(path) < max_len:
                stack.append((neighbor, path + [neighbor]))
    return all_paths


def path_probability(path, adj, Lplus, node_idx):
    edges = [(path[k], path[k + 1]) for k in range(len(path) - 1)]
    m = len(edges)
    if m == 0:
        return 0.0
    K_pi = np.zeros((m, m))
    for a, (ia, ja) in enumerate(edges):
        w_a = adj[ia][ja]
        idx_ia = node_idx[ia]
        idx_ja = node_idx[ja]
        for b, (ib, jb) in enumerate(edges):
            idx_ib = node_idx[ib]
            idx_jb = node_idx[jb]
            Y_ab = (Lplus[idx_ia, idx_ib] + Lplus[idx_ja, idx_jb]
                    - Lplus[idx_ia, idx_jb] - Lplus[idx_ja, idx_ib])
            K_pi[a, b] = w_a * Y_ab
    return np.linalg.det(K_pi)


def compute_participation_ratio(paths, path_probs, source, target):
    if len(paths) == 0 or sum(path_probs) <= 0:
        return 0.0, {}
    Z = sum(path_probs)
    norm_probs = [p / Z for p in path_probs]
    residue_prob = defaultdict(float)
    for path, prob in zip(paths, norm_probs):
        for res in path:
            residue_prob[res] += prob
    if len(residue_prob) == 0:
        return 0.0, {}
    total = sum(residue_prob.values())
    if total <= 0:
        return 0.0, {}
    normalized = {k: v / total for k, v in residue_prob.items()}
    sum_sq = sum(p ** 2 for p in normalized.values())
    PR = 1.0 / sum_sq if sum_sq > 0 else 0.0
    return PR, normalized


def main():
    try:
        from google.colab import files as colab_files
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    pdb_files = {"WT": "6GOD.pdb", "G12D": "6GOF.pdb"}

    for label, fname in pdb_files.items():
        if not os.path.exists(fname):
            if IN_COLAB:
                print(f"Upload {fname}:")
                uploaded = colab_files.upload()
            else:
                raise FileNotFoundError(f"{fname} not found.")

    results = {}
    for label, fname in pdb_files.items():
        print(f"\n{'='*70}")
        print(f"  Processing {label} ({fname})")
        print(f"{'='*70}")

        coords = parse_pdb_ca(fname)
        adj, nodes = build_contact_graph(coords, CUTOFF, KT)
        L, Lplus, node_idx = build_laplacian(adj, nodes)

        print(f"  Residues: {len(coords)}, Graph nodes: {len(nodes)}")
        results[label] = {}

        for source, target, ch_label in CHANNELS:
            print(f"\n  --- {ch_label} ---")
            if source not in node_idx or target not in node_idx:
                print(f"  WARNING: endpoints not in graph!")
                results[label][ch_label] = {'PR': 0.0, 'n_paths': 0, 'n_residues': 0, 'residue_probs': {}}
                continue

            paths = enumerate_paths(adj, source, target, MAX_LEN)
            print(f"  Paths found: {len(paths)}")

            if len(paths) == 0:
                results[label][ch_label] = {'PR': 0.0, 'n_paths': 0, 'n_residues': 0, 'residue_probs': {}}
                continue

            path_probs = []
            for p in paths:
                prob = path_probability(p, adj, Lplus, node_idx)
                path_probs.append(max(prob, 0.0))

            Z_AB = sum(path_probs)
            print(f"  Z_AB = {Z_AB:.6e}")

            PR, res_probs = compute_participation_ratio(paths, path_probs, source, target)
            n_residues = len(res_probs)
            print(f"  PR = {PR:.4f}  (max possible = {n_residues})")

            results[label][ch_label] = {
                'PR': PR, 'n_paths': len(paths), 'n_residues': n_residues,
                'residue_probs': res_probs, 'Z_AB': Z_AB,
            }

    # Summary
    print(f"\n\n{'='*90}")
    print(f"  PARTICIPATION RATIO SUMMARY")
    print(f"{'='*90}")

    for source, target, ch_label in CHANNELS:
        rw = results['WT'].get(ch_label, {'PR': 0, 'n_residues': 0})
        rm = results['G12D'].get(ch_label, {'PR': 0, 'n_residues': 0})
        pr_wt = rw['PR']
        pr_mut = rm['PR']
        delta = pr_mut - pr_wt
        pct = 100 * delta / pr_wt if pr_wt > 0 else 0
        print(f"  {ch_label:<35}  PR_WT={pr_wt:8.3f}  PR_G12D={pr_mut:8.3f}  "
              f"Delta={delta:+8.3f}  ({pct:+.1f}%)")

    outfile = "participation_ratio_results.txt"
    with open(outfile, 'w') as f:
        f.write("PARTICIPATION RATIO RESULTS\n")
        f.write(f"Cutoff = {CUTOFF} A, kT = {KT}, Max path length = {MAX_LEN}\n")
        f.write(f"PR = 1 / sum(p_i^2)\n")
        f.write("=" * 90 + "\n")
        for source, target, ch_label in CHANNELS:
            rw = results['WT'].get(ch_label, {'PR': 0, 'n_residues': 0})
            rm = results['G12D'].get(ch_label, {'PR': 0, 'n_residues': 0})
            f.write(f"{ch_label:<35}  PR_WT={rw['PR']:8.3f}  PR_G12D={rm['PR']:8.3f}  "
                    f"Delta={rm['PR']-rw['PR']:+8.3f}\n")
    print(f"\nSaved: {outfile}")


if __name__ == "__main__":
    main()



  Processing WT (6GOD.pdb)
  Residues: 172, Graph nodes: 172

  --- Ch1: P-loop (6-11) ---
  Paths found: 946282
  Z_AB = 2.787372e-01
  PR = 13.7902  (max possible = 151)

  --- Ch2: pre-Switch II (55-60) ---
  Paths found: 638606
  Z_AB = 3.648228e-01
  PR = 13.5541  (max possible = 131)

  --- Ch3: GBS (110-117) ---
  Paths found: 824774
  Z_AB = 1.056373e-01
  PR = 15.9189  (max possible = 147)

  --- Ch4: SAK motif (141-146) ---
  Paths found: 934827
  Z_AB = 3.283311e-01
  PR = 12.8450  (max possible = 117)

  --- Ch5: Inter-lobe linker (19-142) ---
  Paths found: 864439
  Z_AB = 2.539098e-01
  PR = 15.4478  (max possible = 129)

  Processing G12D (6GOF.pdb)
  Residues: 172, Graph nodes: 172

  --- Ch1: P-loop (6-11) ---
  Paths found: 968240
  Z_AB = 2.762800e-01
  PR = 13.8796  (max possible = 152)

  --- Ch2: pre-Switch II (55-60) ---
  Paths found: 642205
  Z_AB = 3.671149e-01
  PR = 13.4938  (max possible = 132)

  --- Ch3: GBS (110-117) ---
  Paths found: 858454
  Z_AB = 1